# $\sigma_{PS}$ condensate vs mass, all Nf (gsq=8, nt128 L1)

Reads `h0/condensate/etadag_xi` from the massive Real-$m$ disc data
(`data_Nf{nf}_..._mRe{m}..._vmRe{m}.../corr_ylm_disc_tb2_nhits1/`).
Convention = per-mass notebook's $\sigma_{PS}$ cell: $\mathrm{dens}=\mathrm{etadag\_xi}/(N_t\cdot 4\pi)$,
$\sigma_{PS}=\mathrm{dens}+\overline{\mathrm{dens}}$; **contact-subtracted $+2$**
(`condensate_contact_massive_claude.md` Sec 10, mass-independent) = SSB order parameter.
Config-jackknife (1 hit/config).  Light Family-B $m_F=0.2,0.1,0.05,0.01$ (Nf=2,4,6);
**heavy** $m_F=0.40,0.80,1.20$ (Nf=2 only, red diamond; effective $m_F$ at L1).


In [8]:
import h5py, numpy as np, glob, math
import matplotlib.pyplot as plt

Nt = 128; sumA = 4.0*math.pi; norm = Nt*sumA
MASS_DIR = ['0.200000', '0.100000', '0.050000', '0.010000']   # dir string (sea = valence mass)
MASS = np.array([0.2, 0.1, 0.05, 0.01])                       # numeric m_F for the x-axis
NFS = [2, 4, 6]
NFMARK = ['o', 's', '^']                                      # colorblind: distinct marker per Nf

def esn(nf, m): return f'data_Nf{nf}_gsq8.000000at0.200000nu01.000000mRe{m}mIm0.000000nt128L1_vmRe{m}vmIm0.000000/'
def disc_files(nf, m): return sorted(glob.glob(esn(nf, m)+'corr_ylm_disc_tb2_nhits1/corr.*.h0.h5'))

def cond_cfg(files):
    out = []
    for fn in files:
        with h5py.File(fn, 'r') as f:
            out.append(f['h0/condensate/etadag_xi/real'][0] + 1j*f['h0/condensate/etadag_xi/imag'][0])
    return np.array(out)

def jackknife(samp):                  # delete-1 over config axis -> mean, err_re, err_im
    H = samp.shape[0]
    jk = (samp.sum(0)-samp)/(H-1)
    mean = samp.mean(0)
    ere = np.sqrt(np.maximum((H-1)*np.mean((jk.real-mean.real)**2, 0), 0.0))
    eim = np.sqrt(np.maximum((H-1)*np.mean((jk.imag-mean.imag)**2, 0), 0.0))
    return mean, ere, eim

# sigma_PS per (Nf, mass): raw + contact-subtracted (+2), config-jackknifed.  SIG[nf] = (raw_m, raw_e, sub_m, sub_e)
SIG = {}
for nf in NFS:
    rm = []; re = []; sm = []; se = []
    for md_ in MASS_DIR:
        df = disc_files(nf, md_)
        etx = cond_cfg(df)
        dens = etx/norm
        sig = dens + np.conj(dens)
        m0, e0, _ = jackknife(sig)
        m2, e2, _ = jackknife(sig + 2.0)
        rm.append(m0.real); re.append(e0); sm.append(m2.real); se.append(e2)
    SIG[nf] = (np.array(rm), np.array(re), np.array(sm), np.array(se))
    print(f'Nf={nf}: loaded {len(disc_files(nf, MASS_DIR[0]))} cfg/mass')


Nf=2: loaded 140 cfg/mass
Nf=4: loaded 140 cfg/mass
Nf=6: loaded 140 cfg/mass


In [9]:
# Heavy sea masses (Nf2 only): sigma_PS at effective m_F.  Same disc-byproduct condensate,
# contact-sub +2 (mass-independent; condensate_contact_massive_claude.md Sec 10).
# Defensive: skip masses not yet measured (empty / <2 cfg glob).  x-axis = effective m_F (L1).
HEAVY_DIR = ['0.422900', '0.845799', '1.268699']      # dir mRe string (valence = sea = physical m)
HEAVY_MF = [0.40, 0.80, 1.20]                          # effective m_F at L1 (heavy_mass_L124_impl_plan)
hmf = []
hrm = []
hre = []
hsm = []
hse = []
for md_, mf in zip(HEAVY_DIR, HEAVY_MF):
    df = sorted(glob.glob(esn(2, md_) + 'corr_condensate_eo_nhits1/corr.*.h0.h5'))  # NEW dedicated condensate driver (was corr_ylm_disc_tb2)
    if len(df) < 2:
        print(f'heavy m_F={mf}: {len(df)} cfg -- skipped (not yet measured)')
        continue
    etx = cond_cfg(df)
    dens = etx/norm
    sig = dens + np.conj(dens)
    m0, e0, _ = jackknife(sig)
    m2, e2, _ = jackknife(sig + 2.0)
    hmf.append(mf)
    hrm.append(m0.real)
    hre.append(e0)
    hsm.append(m2.real)
    hse.append(e2)
    print(f'heavy m_F={mf}: {len(df)} cfg  sigma_PS(sub +2) = {m2.real:+.5f} +/- {e2:.1e}')
HEAVY = (np.array(hmf), np.array(hrm), np.array(hre), np.array(hsm), np.array(hse))


heavy m_F=0.4: 70 cfg  sigma_PS(sub +2) = +0.29650 +/- 3.3e-05
heavy m_F=0.8: 70 cfg  sigma_PS(sub +2) = +0.52569 +/- 3.7e-05
heavy m_F=1.2: 70 cfg  sigma_PS(sub +2) = +0.70485 +/- 4.5e-05


In [10]:
# Heavy sigma_FS (Nf2): furnished condensate <sigma_FS> = etadag_xi - xidag_1mDdag_eta
# (condensate_contact_massive_claude.md Eq 303), contact-sub +2 - m_F (FS contact = (m_F-2) V_st, Eq 206).
# Reads corr_condensate_eo_nhits1/ (dedicated e/o condensate driver).  Config-jackknife; x = effective m_F.
def cond_fs_cfg(files):
    out = []
    for fn in files:
        with h5py.File(fn, 'r') as f:
            xi  = f['h0/condensate/etadag_xi/real'][0]        + 1j*f['h0/condensate/etadag_xi/imag'][0]
            xid = f['h0/condensate/xidag_1mDdag_eta/real'][0] + 1j*f['h0/condensate/xidag_1mDdag_eta/imag'][0]
            out.append(xi - xid)
    return np.array(out)

hmf_fs = []
hfs_rm = []
hfs_re = []
hfs_sm = []
hfs_se = []
for md_, mf in zip(HEAVY_DIR, HEAVY_MF):
    df = sorted(glob.glob(esn(2, md_) + 'corr_condensate_eo_nhits1/corr.*.h0.h5'))
    if len(df) < 2:
        print(f'heavy FS m_F={mf}: {len(df)} cfg -- skipped')
        continue
    sig = cond_fs_cfg(df)/norm
    m0, e0, _ = jackknife(sig)
    m2, e2, _ = jackknife(sig + (2.0 - mf))
    hmf_fs.append(mf)
    hfs_rm.append(m0.real)
    hfs_re.append(e0)
    hfs_sm.append(m2.real)
    hfs_se.append(e2)
    print(f'heavy FS m_F={mf}: {len(df)} cfg  sigma_FS(sub +2-mF) = {m2.real:+.5f} +/- {e2:.1e}')
HEAVY_FS = (np.array(hmf_fs), np.array(hfs_rm), np.array(hfs_re), np.array(hfs_sm), np.array(hfs_se))

heavy FS m_F=0.4: 70 cfg  sigma_FS(sub +2-mF) = -0.05926 +/- 6.7e-06
heavy FS m_F=0.8: 70 cfg  sigma_FS(sub +2-mF) = -0.21025 +/- 1.5e-05
heavy FS m_F=1.2: 70 cfg  sigma_FS(sub +2-mF) = -0.42289 +/- 2.7e-05


In [11]:
# Table: sigma_PS raw and contact-subtracted (+2), all Nf x all masses.
print(f"{'m_F':>5} {'Nf':>3} {'sigma_PS raw':>22} {'contact-sub (+2)':>22}")
for j, m in enumerate(MASS):
    for nf in NFS:
        rm, re, sm, se = SIG[nf]
        print(f"{m:>5} {nf:>3}   {rm[j]:>+10.5f} +/- {re[j]:.1e}    {sm[j]:>+10.5f} +/- {se[j]:.1e}")
    print()


  m_F  Nf           sigma_PS raw       contact-sub (+2)
  0.2   2     -1.84230 +/- 1.2e-05      +0.15770 +/- 1.2e-05
  0.2   4     -1.84222 +/- 1.1e-05      +0.15778 +/- 1.1e-05
  0.2   6     -1.84217 +/- 1.1e-05      +0.15783 +/- 1.1e-05

  0.1   2     -1.91864 +/- 8.1e-06      +0.08136 +/- 8.1e-06
  0.1   4     -1.91862 +/- 6.6e-06      +0.08138 +/- 6.6e-06
  0.1   6     -1.91859 +/- 7.6e-06      +0.08141 +/- 7.6e-06

 0.05   2     -1.95868 +/- 3.8e-06      +0.04132 +/- 3.8e-06
 0.05   4     -1.95867 +/- 3.5e-06      +0.04133 +/- 3.5e-06
 0.05   6     -1.95865 +/- 3.5e-06      +0.04135 +/- 3.5e-06

 0.01   2     -1.99163 +/- 7.8e-07      +0.00837 +/- 7.8e-07
 0.01   4     -1.99163 +/- 7.2e-07      +0.00837 +/- 7.2e-07
 0.01   6     -1.99163 +/- 8.3e-07      +0.00837 +/- 8.3e-07



In [ ]:
# Contact-subtracted sigma_PS vs m_F, LOG-LOG: light Family-B (Nf=2,4,6) + heavy (Nf=2, red diamond).
# NOTE (unresolved convention mismatch at L1): light x = physical bare mass (factor 1.0), while heavy
# x = physical x 0.9459 (L1 measure factor).  Slopes are not directly comparable until light is put on
# the same measure factor (cf. the L2 notebook, which was reconciled to physical x 0.506305).
# Light linear chiral fit (intercept = chiral limit) printed before the plot.
for nf in NFS:
    rm, re, sm, se = SIG[nf]
    slope, icpt = np.polyfit(MASS, sm, 1)
    print(f'Nf={nf}: contact-sub sigma_PS = {slope:+.4f}*m_F + ({icpt:+.5f})   [linear fit; intercept = m->0 condensate]')

fig, ax = plt.subplots()
mg = np.logspace(np.log10(MASS.min()*0.5), np.log10(MASS.max()*1.1), 200)
for i, nf in enumerate(NFS):
    rm, re, sm, se = SIG[nf]
    slope, icpt = np.polyfit(MASS, sm, 1)
    ax.errorbar(MASS, sm, yerr=se, marker=NFMARK[i], ls='none', capsize=3, label=f'Nf={nf}')
    ax.plot(mg, slope*mg + icpt, ls='--', lw=0.8, color=ax.lines[-1].get_color())
hmf, hrm, hre, hsm, hse = HEAVY
if len(hmf):
    ax.errorbar(hmf, hsm, yerr=hse, marker='D', color='red', ls='none', capsize=3, label='Nf=2 heavy')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$m_F$'); ax.set_ylabel(r'$\sigma_{PS}$ (contact-subtracted, $+2$)')
ax.set_title(r'$\sigma_{PS}$ contact-subtracted vs $m_F$ (gsq=8, L1, log-log): Nf=2,4,6 + heavy Nf2')
ax.legend(); plt.tight_layout()


In [ ]:
# Contact-subtracted sigma_FS vs m_F, LOG-LOG (heavy Nf2, red diamond).  sigma_FS is uniformly
# NEGATIVE here, so we plot -sigma_FS on log-y.  Order param = sigma_FS/V_st + (2 - m_F).
fig, ax = plt.subplots()
hmf_fs, hfs_rm, hfs_re, hfs_sm, hfs_se = HEAVY_FS
if len(hmf_fs):
    ax.errorbar(hmf_fs, -hfs_sm, yerr=hfs_se, marker='D', color='red', ls='none', capsize=3, label='Nf=2 heavy')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$m_F$')
ax.set_ylabel(r'$-\sigma_{FS}$ (contact-subtracted, $+2-m_F$)')
ax.set_title(r'$-\sigma_{FS}$ contact-subtracted vs $m_F$ (gsq=8, L1, log-log): heavy Nf2')
ax.legend()
plt.tight_layout()

In [14]:
# # Raw sigma_PS vs m_F, Nf=2,4,6 (before the +2 contact subtraction).
# fig, ax = plt.subplots()
# for i, nf in enumerate(NFS):
#     rm, re, sm, se = SIG[nf]
#     ax.errorbar(MASS, rm, yerr=re, marker=NFMARK[i], ls='none', capsize=3, label=f'Nf={nf}')
# ax.axhline(-2.0, color='k', lw=0.5, ls=':')
# ax.set_xlabel(r'$m_F$'); ax.set_ylabel(r'$\sigma_{PS}$ (raw)')
# ax.set_title(r'$\sigma_{PS}$ raw vs $m_F$ (gsq=8): Nf=2,4,6  (contact term $\approx-2$)')
# ax.legend(); plt.tight_layout()
